In [1]:
import pandas as pd

dataset_path = r'C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa'
df = pd.read_parquet(dataset_path + '/domain_users.parquet')
print(df.head())
print(df)

          group username
0  sanfrancisco  mat.sco
1  sanfrancisco  mat.sco
2  sanfrancisco  mat.sco
3  sanfrancisco  mat.sco
4  sanfrancisco  mat.sco
            group        username
0    sanfrancisco         mat.sco
1    sanfrancisco         mat.sco
2    sanfrancisco         mat.sco
3    sanfrancisco         mat.sco
4    sanfrancisco         mat.sco
5    sanfrancisco      flet.lee33
6    sanfrancisco      flet.lee33
7    sanfrancisco  maddi.campbel8
8    sanfrancisco  maddi.campbel8
9    sanfrancisco         ricwood
10   sanfrancisco         ricwood
11   sanfrancisco         ricwood
12   sanfrancisco         ricwood
13  domain admins         mat.sco
14  domain admins   Administrator
15  domain admins          svc_em


In [2]:
import os

for root, dirs, files in os.walk(dataset_path):
    for f in files:
        print(os.path.join(root, f))

C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\domain_users.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\loggedonusers\CRW167SL.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\loggedonusers\PXA949WW.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\loggedonusers\RDL208NS.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\loggedonusers\RDL794WW.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\loggedonusers\UMI895SB.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\useraccounts\CRW167SL.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\useraccounts\PXA949WW.parquet
C:\Users\nahla\Desktop\threat_hunt_task\003d1fc2-1f7e-4a11-a011-b684a313f9fa\useraccounts\RDL208NS.parquet
C:\Users\nahla\Desktop\threat_hunt_task\0

In [3]:
import glob

def load_all(folder_name):
    files = glob.glob(dataset_path + f'/{folder_name}/*.parquet')
    dfs = []
    for f in files:
        d = pd.read_parquet(f)
        d['hostname'] = os.path.basename(f).replace('.parquet', '')
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [4]:
processes = load_all('w32processes')
processes_memory = load_all('w32processes-memorysections')
services = load_all('w32services')
tasks = load_all('w32tasks')
drivers = load_all('w32drivers')
persistence_reg = load_all('w32persistence-registryitems')
persistence_files = load_all('w32persistence-fileitems')
persistence_svc = load_all('w32persistence-serviceitems')
logged_users = load_all('loggedonusers')
user_accounts = load_all('useraccounts')

print(processes.shape, processes_memory.shape, services.shape, tasks.shape)
print(sorted(processes['hostname'].unique()))

(461, 6) (18467, 9) (2183, 14) (604, 14)
['CRW167SL', 'PXA949WW', 'RDL208NS', 'RDL794WW', 'UMI895SB']


In [5]:
domain_users = pd.read_parquet(dataset_path + '/domain_users.parquet')
domain_users

,group,username
0,sanfrancisco,mat.sco
1,sanfrancisco,mat.sco
2,sanfrancisco,mat.sco
3,sanfrancisco,mat.sco
4,sanfrancisco,mat.sco
5,sanfrancisco,flet.lee33
6,sanfrancisco,flet.lee33
7,sanfrancisco,maddi.campbel8
8,sanfrancisco,maddi.campbel8
9,sanfrancisco,ricwood


In [6]:
print(logged_users.columns.tolist())
logged_users

['domain-user', 'hostname', 'local-user', 'username']


,domain-user,hostname,local-user,username
0,False,CRW167SL,True,Registry
1,False,CRW167SL,True,smss.exe
2,False,CRW167SL,True,lsass.exe
3,False,CRW167SL,True,svchost.exe
4,False,CRW167SL,True,services.exe
...,...,...,...,...
225,False,UMI895SB,True,vmware-authd.exe
226,False,UMI895SB,True,armsvc.exe
227,False,UMI895SB,True,AcroRd32.exe
228,False,UMI895SB,True,RdrCEF.exe


In [7]:
logged_users[logged_users.astype(str).apply(
    lambda row: row.str.contains('admin', case=False, na=False)
).any(axis=1)]

,domain-user,hostname,local-user,username
189,True,RDL794WW,False,Administrator


In [8]:
system_procs = processes[processes['username'].astype(str).str.contains('SYSTEM', case=False, na=False)]
print(system_procs.shape)
system_procs[['hostname','name','path','pid']]

(372, 6)


,hostname,name,path,pid
0,CRW167SL,svchost.exe,c:\windows\system32,4824
1,CRW167SL,svchost.exe,c:\windows\system32,748
2,CRW167SL,svchost.exe,c:\windows\system32,1084
5,CRW167SL,dwm.exe,C:\Windows\System32,1220
6,CRW167SL,ctfmon.exe,C:\Windows\system32,5140
...,...,...,...,...
452,UMI895SB,svchost.exe,C:\Windows\system32,2884
455,UMI895SB,svchost.exe,C:\Windows\system32,1884
456,UMI895SB,Dwm.exe,C:\Windows\system32,5196
458,UMI895SB,conhost.exe,C:\Windows\system32,1928


In [9]:
system_procs[system_procs.astype(str).apply(
    lambda row: row.str.contains(r'Users\\', case=False, na=False)
).any(axis=1)]

,arguments,hostname,name,path,pid,username


In [10]:
suspicious = processes[processes.astype(str).apply(
    lambda row: row.str.contains('Temp|AppData|ProgramData|WindowsApps', case=False, na=False)
).any(axis=1)]
suspicious[['hostname','name','path','username']]

,hostname,name,path,username
11,CRW167SL,OneDrive.exe,C:\Users\mat.sco\AppData\Local\Microsoft\OneDrive,mat.sco
15,CRW167SL,GitHubDesktop.exe,C:\Users\ricwood\AppData\Local\GitHubDesktop\a...,ricwood
76,CRW167SL,GitHubDesktop.exe,C:\Users\ricwood\AppData\Local\GitHubDesktop\a...,ricwood
95,CRW167SL,SkypeBackgroundHost.exe,C:\Program Files\WindowsApps\Microsoft.SkypeAp...,mat.sco
194,PXA949WW,OneDrive.exe,C:\Users\flet.lee33\AppData\Local\Microsoft\On...,flet.lee33
208,PXA949WW,g2mcomm.exe,C:\Users\maddi.campbel8\AppData\Local\GoToMeet...,maddi.campbel8
215,PXA949WW,g2mstart.exe,C:\Users\maddi.campbel8\AppData\Local\GoToMeet...,maddi.campbel8
242,PXA949WW,SkypeBackgroundHost.exe,C:\Program Files\WindowsApps\Microsoft.SkypeAp...,flet.lee33
265,RDL208NS,GitHubDesktop.exe,C:\Users\dani.robi52\AppData\Local\GitHubDeskt...,dani.robi52
327,RDL208NS,GitHubDesktop.exe,C:\Users\dani.robi52\AppData\Local\GitHubDeskt...,dani.robi52


In [11]:
for h in ['CRW167SL', 'PXA949WW', 'RDL208NS', 'RDL794WW', 'UMI895SB']:
    print(f"--- {h} ---")
    print(processes[processes['hostname'] == h][['name','path','username']].to_string())
    print()

--- CRW167SL ---
                          name                                                                           path             username
0                  svchost.exe                                                            c:\windows\system32  NT AUTHORITY\SYSTEM
1                  svchost.exe                                                            c:\windows\system32  NT AUTHORITY\SYSTEM
2                  svchost.exe                                                            c:\windows\system32  NT AUTHORITY\SYSTEM
3                     mbam.exe                                     C:\Program Files\Malwarebytes\Anti-Malware              mat.sco
4                      cmd.exe                                                    C:\Windows\System32\cmd.exe              mat.sco
5                      dwm.exe                                                            C:\Windows\System32  NT AUTHORITY\SYSTEM
6                   ctfmon.exe                                    

In [12]:
logged_users

,domain-user,hostname,local-user,username
0,False,CRW167SL,True,Registry
1,False,CRW167SL,True,smss.exe
2,False,CRW167SL,True,lsass.exe
3,False,CRW167SL,True,svchost.exe
4,False,CRW167SL,True,services.exe
...,...,...,...,...
225,False,UMI895SB,True,vmware-authd.exe
226,False,UMI895SB,True,armsvc.exe
227,False,UMI895SB,True,AcroRd32.exe
228,False,UMI895SB,True,RdrCEF.exe


In [13]:
user_accounts

,domain-admin,domain-user,hostname,local-admin,local-user,username
0,False,True,CRW167SL,False,False,ricwood
1,False,True,CRW167SL,False,False,mat.sco
2,False,False,CRW167SL,False,True,elijgonzal
3,False,False,CRW167SL,True,True,zBMIahqYKPEfTT
4,False,True,PXA949WW,False,False,flet.lee33
5,False,True,PXA949WW,True,False,ricwood
6,False,True,PXA949WW,False,False,maddi.campbel8
7,True,True,PXA949WW,False,False,Administrator
8,False,False,PXA949WW,True,True,dEkSarQLRvDfhPA
9,False,True,RDL208NS,True,False,mat.sco


In [14]:
print(processes['name'].value_counts())

name
svchost.exe        235
fontdrvhost.exe      6
RdrCEF.exe           6
AcroRd32.exe         6
mbam.exe             5
                  ... 
soffice.bin          1
lsm.exe              1
taskhost.exe         1
soffice.exe          1
Dwm.exe              1
Name: count, Length: 80, dtype: int64


In [15]:
tasks['hostname'].value_counts()

hostname
CRW167SL    145
PXA949WW    145
RDL208NS    144
RDL794WW    126
UMI895SB     44
Name: count, dtype: int64

In [16]:
for h in ['CRW167SL', 'PXA949WW', 'RDL208NS', 'RDL794WW', 'UMI895SB']:
    t = tasks[tasks['hostname'] == h]
    print(f"--- {h}: {len(t)} tasks ---")

--- CRW167SL: 145 tasks ---
--- PXA949WW: 145 tasks ---
--- RDL208NS: 144 tasks ---
--- RDL794WW: 126 tasks ---
--- UMI895SB: 44 tasks ---


In [17]:
user_accounts[user_accounts['username'].str.contains('zBMI|dEkS', case=False, na=False)]

,domain-admin,domain-user,hostname,local-admin,local-user,username
3,False,False,CRW167SL,True,True,zBMIahqYKPEfTT
8,False,False,PXA949WW,True,True,dEkSarQLRvDfhPA


In [18]:
for h in ['CRW167SL', 'PXA949WW']:
    print(f"=== {h} — Registry Persistence ===")
    print(persistence_reg[persistence_reg['hostname'] == h].to_string())
    print(f"=== {h} — File Persistence ===")
    print(persistence_files[persistence_files['hostname'] == h].to_string())

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



   drive fileextension                         filename                                                          filepath                                                                          fullpath  hostname                     username
12     c           exe                  mbamservice.exe                           program files\malwarebytes\anti-malware                        c:\program files\malwarebytes\anti-malware\mbamservice.exe  PXA949WW       BUILTIN\Administrators
13     c           dll  onedrivesettingsyncprovider.dll                                                  Windows\System32                               c:\Windows\System32\onedrivesettingsyncprovider.dll  PXA949WW  NT SERVICE\TrustedInstaller
14     c           exe                onedrivesetup.exe                                                  Windows\SysWOW64                                             c:\Windows\SysWOW64\onedrivesetup.exe  PXA949WW  NT SERVICE\TrustedInstaller
15     c           exe      

In [19]:
for h in ['CRW167SL', 'PXA949WW']:
    print(f"=== {h} — All Processes ===")
    print(processes[processes['hostname'] == h][['name','path','username']].to_string())

=== CRW167SL — All Processes ===
                          name                                                                           path             username
0                  svchost.exe                                                            c:\windows\system32  NT AUTHORITY\SYSTEM
1                  svchost.exe                                                            c:\windows\system32  NT AUTHORITY\SYSTEM
2                  svchost.exe                                                            c:\windows\system32  NT AUTHORITY\SYSTEM
3                     mbam.exe                                     C:\Program Files\Malwarebytes\Anti-Malware              mat.sco
4                      cmd.exe                                                    C:\Windows\System32\cmd.exe              mat.sco
5                      dwm.exe                                                            C:\Windows\System32  NT AUTHORITY\SYSTEM
6                   ctfmon.exe                    

In [20]:
for h in ['CRW167SL', 'PXA949WW']:
    print(f"=== {h} — Services ===")
    print(services[services['hostname'] == h].to_string())

=== CRW167SL — Services ===
                                                                                                    arguments                                                                                                                                                                                                                                                                                                                                                      description                                                              descriptivename  hostname                               md5                          name                                                                                                     path                            pathcertificateissuer                              pathcertificatesubject                            pathsignaturedescription pathsignatureexists pathsignatureverified                                               servicedll  

In [21]:
processes[processes['username'].astype(str).str.contains('svc_em', case=False, na=False)]

,arguments,hostname,name,path,pid,username


In [22]:
services[services.astype(str).apply(lambda row: row.str.contains('svc_em', case=False, na=False)).any(axis=1)]

,arguments,description,descriptivename,hostname,md5,name,path,pathcertificateissuer,pathcertificatesubject,pathsignaturedescription,pathsignatureexists,pathsignatureverified,servicedll,type


In [23]:
persistence_files[persistence_files.astype(str).apply(lambda row: row.str.contains('Public', case=False, na=False)).any(axis=1)]

,drive,fileextension,filename,filepath,fullpath,hostname,username
1578,c,dll,mbaeapipublic.dll,Windows\SysWOW64,c:\Windows\SysWOW64\mbaeapipublic.dll,RDL794WW,NT AUTHORITY\SYSTEM
2150,c,dll,mbaeapipublic.dll,Windows\System32,c:\Windows\System32\mbaeapipublic.dll,RDL794WW,NT AUTHORITY\SYSTEM


In [24]:
logged_users[logged_users['username'].isin(['zBMIahqYKPEfTT', 'dEkSarQLRvDfhPA'])]

,domain-user,hostname,local-user,username


In [30]:
## Findings Summary

###**Two compromised machines identified: CRW167SL and PXA949WW**

### CRW167SL

#### **Randomly-generated local admin account**: A local account named `zBMIahqYKPEfTT` was found with `local-admin = True`. This naming pattern (random alphanumeric string) is inconsistent with the organization's normal username convention (e.g. `firstname.lastname`) and is a strong indicator of an attacker-created backdoor account.

#### **Malicious service masquerading (hidinterrupt)**: A service named `hidinterrupt` was found pointing to `C:\Public\run.vbs`. On a comparison host (PXA949WW), the legitimate service with the same name (`hidinterrupt`) correctly points to `C:\Windows\System32\drivers\hidinterrupt.sys`, a genuine Windows kernel driver. Replacing a trusted system service's target with a `.vbs` script in a non-standard, world-writable location (`C:\Public`) is a classic service masquerading / persistence technique used to blend malicious code in with legitimate-looking system services.

### PXA949WW

#### **Randomly-generated local admin account**: A local account named `dEkSarQLRvDfhPA` was found with `local-admin = True`, following the same suspicious random-string naming pattern observed on CRW167SL.

### Conclusion

#### Both `CRW167SL` and `PXA949WW` show independent, host-specific indicators of compromise not present on the other three hosts in the dataset (`RDL208NS`, `RDL794WW`, `UMI895SB`). The presence of randomly-named local administrator accounts on both machines, combined with a confirmed malicious service masquerading as a legitimate Windows driver on `CRW167SL`, provides strong evidence that these two hosts were compromised — most likely through the creation of hidden backdoor accounts and a service-based persistence mechanism.

#### (See the accompanying report for the Executive Summary, IOC explanations in plain language, recommendations, limitations, and assumptions.)*